[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/weighted_dual/weighted_dual.ipynb)

# Weighted Dual Meshes on the Sphere

This example computes the polar dual of a polyhedron about the unit sphere, and how it moves as the polyhedron's vertices slide along their rays from the centre. Seen from the centre, every vertex has an image on the sphere: the polyhedron's images make a triangle mesh on the sphere, and the images of the dual's vertices make its dual mesh. How far a point lies from the centre does not move its image, but it tilts the faces around it, and with them the dual's vertices: the distances shape the dual mesh, which the mesh on the sphere does not show. The points first breathe in and out along fixed directions; then a tangled dual mesh is untangled by the nearest distances that make every dual edge positive.

Points are antivectors and planes are vectors. A face lies in the plane that pairs to one with each of its vertices, `plane & vertex == 1`, and polarity about the unit sphere is the dual: the face's pole is `plane.dual()`, a point. Whether a pole lies inside its face, and how long a dual edge is, are pairings of planes through the centre with points, each signed against a corner of the mesh, so neither the faces nor space need a hand. Along a fixed ray a point is fixed by its weight, one over its distance, and the folds of the polyhedron and the positions of the poles are linear in the weights.

In the notation of discrete exterior calculus, the dual edge lengths over the primal ones are the diagonal Hodge star on edges, $\star_1$, and the mesh of a hull of points at varying distances is a weighted Delaunay triangulation of their images on the sphere, with its dual mesh the power diagram.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from scipy.optimize import LinearConstraint, linprog, minimize

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.weighted_dual import render
from examples.surfaces.weighted_dual.core import hull, timing

mv = NumpyContext(ga).multivector
# Mesh topology: edges, and the faces and vertices either side of each.
Mesh = instantiate("examples.util.mesh", ga).Mesh
Scalar = ga.gatype.scalar()                         # Scalar+
# Points, and directions from the centre, are antivectors.
Point = ga.gatype.antivector()                      # Antivector-


## 1. Faces and poles

A face's plane is the one that pairs to one with each of its vertices: the planes through the centre and each of its edges, summed, over the volume of its corners, `((a & b) + (b & c) + (c & a)) * (a & b & c).inverse()`. Its dual about the unit sphere, `planes.dual()`, is the face's pole, and the poles are the vertices of the polar dual. On the sphere a pole lands inside its face's image when it lies on the face's side of each of the three planes through the centre and an edge, as the face's third corner does: the two pairings with each such plane, the pole's and the third corner's, agree in sign. The convex hull itself, which triangles the points make, is one step outside the algebra, taken by scipy; the linear programs of the last section are the other.

In [ ]:
# The points, how far their directions are nudged off an even spread, and the seed of the nudges.
n_points = 32
jitter = 0.03
direction_seed = 0

In [ ]:
# Directions spread evenly over the sphere along a Fibonacci spiral, each nudged a little at random: the pole
# turned down towards x by its polar angle, then around the pole by its azimuth.
turns = np.arange(n_points) + 0.5
polar = np.arccos(1 - 2 * turns / n_points)
azimuth = np.pi * (1 + 5**0.5) * turns
down = (mv.zx * (-polar / 2)).exp()                                            # [points] Rotor+
around = (mv.xy * (-azimuth / 2)).exp()                                        # [points] Rotor+
spiral = (around * down) >> mv.z.dual()                                        # [points] Point-
nudges = mv.vector(np.random.default_rng(direction_seed).normal(size=(n_points, 3))).dual()   # [points] Point-
directions = (spiral + jitter * nudges).normalized()                           # [points] Point-


def poles(points: Point, faces: np.ndarray) -> Point:
    """The pole of each face about the unit sphere is the dual of the face's plane, the plane that pairs to one
    with each of its vertices: the planes through the centre and each edge, summed, over the volume of the
    vertices."""
    a, b, c = (points[..., faces[:, corner]] for corner in range(3))           # [..., faces] Point-, Point-, Point-
    planes = ((a & b) + (b & c) + (c & a)) * (a & b & c).inverse()             # [..., faces] Plane+
    return planes.dual()                                                       # [..., faces] Point-


def sides(points: Point, faces: np.ndarray, duals: Point) -> Scalar:
    """Each of a face's three sides is positive where the face's pole, seen from the centre, lies on the face's
    side of the plane through the centre and an edge, as the face's third vertex does; all three are positive
    when the pole lands inside the face."""
    corners = points[..., faces]                                               # [..., faces, 3] Point-
    walls = corners & corners[..., [1, 2, 0]]                                  # [..., faces, 3] Plane-
    return (walls & duals[..., None]) * (walls & corners[..., [2, 0, 1]])      # [..., faces, 3] Scalar+


def dual_lengths(points: Point, mesh: Mesh) -> Scalar:
    """A dual edge's signed length, seen from the centre, is how much further the pole of the face on the right
    of its edge lies to that side than the pole of the face on the left; it is negative where the two poles have
    crossed over."""
    edges = mesh.edges[mesh.interior_edges]                                    # [edges, 2]
    left, right = mesh.edge_faces.T                                            # [edges], [edges]
    duals = poles(points, mesh.faces)                                          # [..., faces] Point-
    wall = points[..., edges[:, 0]] & points[..., edges[:, 1]]                 # [..., edges] Plane-
    toward = wall & points[..., mesh.edge_opposites[:, 1]]                     # [..., edges] Scalar-
    return (wall & (duals[..., right] - duals[..., left])) * toward / toward.norm()   # [..., edges] Scalar+


def folds(directions: Point, weights: np.ndarray, mesh: Mesh) -> Scalar:
    """An edge's fold is how far it bends outward, as on a convex polyhedron, or inward where negative, for
    points along the directions at the distances one over their weights: the volume of the edge's ends and the
    two vertices across it, times their four weights. That is a sum over the four corners of each weight times
    the volume of the other three directions, with alternating signs, so it is linear in the weights, which lie
    on the last axis. Each fold is scaled by the volume of the edge's ends and the vertex across it on the left,
    which sets the sense of the centre."""
    # Each edge's ends, then the vertices across it on its left and on its right.
    corners = np.concatenate([mesh.edges[mesh.interior_edges], mesh.edge_opposites], axis=-1)   # [edges, 4]
    others = corners[:, [[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]]]          # [edges, 4, 3]
    volumes = directions[others[..., 0]] & directions[others[..., 1]] & directions[others[..., 2]]   # [edges, 4] Scalar-
    face = volumes[:, 3]                                                       # [edges] Scalar-
    return (volumes * (weights[..., corners] * [-1, 1, -1, 1])).sum(axis=-1) * face   # [..., edges] Scalar+


def insides(directions: Point, weights: np.ndarray, faces: np.ndarray) -> Scalar:
    """The signed distances of a face's pole inside the planes through the centre and each of its edges are
    positive on the face's side, all three when the pole lands inside the face. The face's plane is the corners'
    weights times the face's reciprocal frame, the three planes that pair to one with their own corner's
    direction and to zero with the other two, and the planes through the edges are fixed by the directions, so
    the distances are linear in the weights, which lie on the last axis."""
    a, b, c = (directions[faces[:, corner]] for corner in range(3))           # [faces] Point-, Point-, Point-
    volume = (a & b & c).inverse()                                             # [faces] Scalar-
    frame = stack([(b & c) * volume, (c & a) * volume, (a & b) * volume], axis=-1)   # [faces, 3] Plane+
    duals = (frame * weights[..., faces]).sum(axis=-1).dual()                  # [..., faces] Point-
    corners = directions[faces]                                                # [faces, 3] Point-
    walls = (corners & corners[:, [1, 2, 0]]).normalized()                     # [faces, 3] Plane-
    third = walls & corners[:, [2, 0, 1]]                                      # [faces, 3] Scalar-
    return (walls & duals[..., None]) * third / third.norm()                   # [..., faces, 3] Scalar+


## 2. Changing the distances

The points on the sphere keep their mesh there, the spherical Delaunay mesh, throughout: its triangles never change. Each point keeps its direction and moves in and out along it, each with its own phase, so its image on the sphere never moves either. What moves is everything the distances shape: the faces tilt, the polyhedron may stop being convex, and the poles of the faces, the dual vertices, travel across the sphere; once the distances differ the dual is no longer the circumcentric one, and need not even sit well inside its triangles. That freedom, invisible in the mesh on the sphere, is the projective degree of freedom of the dual. The left panel is space: the polyhedron, solid and blue, its vertices coloured by their distance from the sphere, blue inside and red outside, and the edges of its polar dual, orange, on the dual's faces turned toward the viewer. The right panel is the half of the sphere facing the viewer, with the images of both meshes.

In [ ]:
# The seed of the points' phases, their spread in distance from the centre, and the frames of one breath
# and the duration of each.
phase_seed = 3
amplitude = 0.06
n_frames = 96
duration_ms = 40

In [ ]:
def breath() -> Iterator[tuple[Point, np.ndarray, Point]]:
    """The breath yields, at each frame, the points along their fixed directions at their own distances, the
    faces of the mesh they had on the sphere, held throughout, and the poles of those faces."""
    faces = hull(directions)                                                   # [faces, 3]
    phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, n_points)
    radii = 1 + amplitude * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None] + phases)   # [frames, points]
    for points in directions * radii:                                          # [points] Point-
        yield points, faces, poles(points, faces)


frames = list(breath())
well_centred = np.array([(sides(points, faces, duals) > 0).all(axis=-1).sum() for points, faces, duals in frames])   # [frames]


In [ ]:
print(f"well centred faces over the breath: {well_centred.min()} to {well_centred.max()} of {len(frames[0][1])}")
save_figure(render.draw_complex(*frames[0]), "weighted_dual");

In [ ]:
show_animation(render.animate(frames), "weighted_dual_breath", duration_ms)

## 3. Weights that keep every dual edge positive

The hull of the points at random distances along their rays has a mesh of its own. With every point put back on the sphere the images stay where they were, but the mesh is no longer the sphere's own, and its dual tangles: the poles of the two faces beside an edge land on the wrong sides of it, a dual edge of negative length, exactly where the polyhedron folds inward. Negative dual lengths are what break the operators built on a dual, such as a Hodge star with negative entries.

Along a fixed direction a point is fixed by its weight, one over its distance from the centre, and in the weights every edge's fold is linear: the volume of the edge's ends and the two vertices across it, times their four weights. So is each pole's signed distance inside its face, since the face's plane is the corners' weights times its reciprocal frame. The weights that unfold the whole mesh with every pole inside its face therefore form a convex cone, and finding them is a linear program, the other step outside the algebra. The widest margin, the largest smallest condition over all weightings, says whether a rescue exists at all; the rescue shown is the nearest one to the tangled start that keeps a third of that margin, which shares the change among many points. Every condition being linear, each crosses zero at most once along the straight line from the start to the rescue: the polyhedron unfolds steadily, edge by edge, and closes in on the rescue over the last quarter of the frames, while the mesh on the sphere stays exactly where it was. The vertices are coloured by their distance from the sphere, blue inside and red outside, each on a stretch of the ray it moves along.

In [ ]:
# The spread and seed of the random distances whose hull gives the mesh the rescue starts from, every point
# on the hull; the bounds on the weights, the share of the widest margin of its conditions the rescue keeps,
# and the frames from the start to the rescue.
mesh_spread, mesh_seed = 0.06, 57
weight_bounds = (0.5, 2.0)
margin_share = 0.3
n_settle_frames = 96

In [ ]:
def widest(conditions: Scalar) -> tuple[np.ndarray, float]:
    """The widest margin is the largest smallest condition, each scaled to unit size, over the weights within
    their bounds and summing to their count; it is negative when no weights meet every condition. A linear
    program finds it, with its weights, outside the algebra."""
    # The conditions as numbers for the solver, one row per condition and one column per point.
    matrix = conditions.to_array().T                                           # [conditions, points]
    scaled = matrix / np.linalg.norm(matrix, axis=-1, keepdims=True)
    count = matrix.shape[-1]
    # The weights, then the smallest condition, which is maximised.
    largest = np.concatenate([np.zeros(count), [-1.0]])
    total = np.concatenate([np.ones(count), [0.0]])[None]
    result = linprog(largest, A_ub=np.hstack([-scaled, np.ones((len(scaled), 1))]), b_ub=np.zeros(len(scaled)),
                     A_eq=total, b_eq=[count], bounds=[weight_bounds] * count + [(None, None)])
    return result.x[:-1], result.x[-1]


def nearest(conditions: Scalar, margin: float, start: np.ndarray) -> np.ndarray:
    """The nearest weights to the start, by the summed squares of the changes, leave every condition, scaled to
    unit size, at least the given margin, which shares the change among many points rather than a few. A
    quadratic program finds them outside the algebra."""
    # The conditions as numbers for the solver, one row per condition and one column per point.
    matrix = conditions.to_array().T                                           # [conditions, points]
    scaled = matrix / np.linalg.norm(matrix, axis=-1, keepdims=True)
    result = minimize(lambda weights: ((weights - start) ** 2).sum(), start, jac=lambda weights: 2 * (weights - start),
                      method="SLSQP", constraints=[LinearConstraint(scaled, margin, np.inf)], bounds=[weight_bounds] * len(start))
    return result.x


In [ ]:
radii = np.exp(mesh_spread * np.random.default_rng(mesh_seed).normal(size=n_points))   # [points]
rescued_mesh = Mesh(n_points, hull(directions * radii))
start = np.ones(n_points)                                                      # [points]
# Every fold outward and every pole inside its face. Each is linear in the weights, so at one unit weighting per
# point, all as one batch, they give the columns of the map from weights to conditions.
unit_weights = np.eye(n_points)                                                # [points, points]
fold_conditions = folds(directions, unit_weights, rescued_mesh)               # [points, edges] Scalar+
inside_conditions = insides(directions, unit_weights, rescued_mesh.faces).reshape(n_points, -1)   # [points, faces * 3] Scalar+
conditions = concatenate([fold_conditions, inside_conditions], axis=-1)        # [points, conditions] Scalar+
_, widest_margin = widest(conditions)
rescue = nearest(conditions, margin_share * widest_margin, start)              # [points]
paths = start + timing(n_settle_frames)[:, None] * (rescue - start)            # [frames, points]
settled = [(points, rescued_mesh.faces, poles(points, rescued_mesh.faces)) for points in directions / paths]


In [ ]:
for name, (points, faces, duals) in (("at the start", settled[0]), ("rescued", settled[-1])):
    print(f"{name:13s} negative dual edges {(dual_lengths(points, rescued_mesh) < 0).sum()} of {len(rescued_mesh.edges)},"
          f" well-centred faces {(sides(points, faces, duals) > 0).all(axis=-1).sum()} of {len(faces)}")
print(f"widest margin of the conditions {widest_margin:.4f}")
show_animation(render.animate(settled), "weighted_dual_rescue", duration_ms)


In [ ]:
# checks
# On the sphere the dual edges cross over exactly where the polyhedron folds; along the straight line to
# the rescue they only ever uncross, and at the rescue none is left.
crossed = [(dual_lengths(points, rescued_mesh) < 0).sum() for points, _, _ in settled]
assert np.array_equal(dual_lengths(settled[0][0], rescued_mesh) < 0, folds(directions, np.ones(n_points), rescued_mesh) < 0)
assert crossed[0] > 0 and crossed[-1] == 0 and (np.diff(crossed) <= 0).all()
# At the rescue every pole lies inside its face.
assert (sides(*settled[-1]) > 0).all()
for points, faces, duals in frames:
    # Every face's plane, the dual of its pole, pairs to one with each of the face's vertices.
    assert np.abs((duals.dual_inverse()[:, None] & points[faces]).to_array() - 1).max() < 1e-10


def flip(mesh: Mesh, edge: int) -> Mesh:
    """Turning one interior edge to join the two vertices across it gives the flipped mesh."""
    left, right = mesh.edge_faces[edge]
    a, b = mesh.edges[mesh.interior_edges[edge]]
    across_left, across_right = mesh.edge_opposites[edge]
    # The face on the left runs along the edge, (a, b, across_left); the face on the right against it.
    faces = mesh.faces.copy()
    faces[left], faces[right] = (a, across_right, across_left), (across_right, b, across_left)
    return Mesh(mesh.n_vertices, faces)


# A mesh that is no hull at any distances has no rescue, and the same program says so: the sphere's mesh
# with so many random edges flipped, from this seed, has a negative widest margin.
n_flips, flip_seed = 8, 6
rng = np.random.default_rng(flip_seed)
flipped_mesh = Mesh(n_points, hull(directions))
for _ in range(n_flips):
    flipped_mesh = flip(flipped_mesh, rng.integers(len(flipped_mesh.interior_edges)))
assert widest(folds(directions, np.eye(n_points), flipped_mesh))[1] < 0